# **Scattering correction for infrared spectra of biological cells using computational infrared microspectroscopy and deep learning**

*Sergio G. Rodrigo $^{1,2*}$, Ilia L. Rasskazov $^{3}$, Luis Martin-Moreno $^{1,4}$, Martin Schnell $^{5,6,7*}$*



+ 1 Instituto de Nanociencia y Materiales de Aragón (INMA), CSIC-Universidad de Zaragoza, Zaragoza, 50009, Spain
+ 2 Departamento de Física Aplicada, Universidad de Zaragoza, Zaragoza, 50009, Spain
+ 3 Independent Researcher, San Jose, California 95124, United States
+ 4 Departamento de Física de la Materia Condensada, Universidad de Zaragoza, Zaragoza, 50009, Spain
+ 5 CIC nanoGUNE BRTA, 20018 Donostia-San Sebastián, Spain
+ 6 Donostia International Physics Center, Donostia-San Sebastian 20018, Spain
+ 7 IKERBASQUE, Basque Foundation for Science, 48013 Bilbao, Spain

$^*$ Corresponding authors. Email: sergut@unizar.es (SGR), schnelloptics@gmail.com (MS)


# *Description*

The inverse design process is based on a gradient-descent algorithm, which goal is to find the optimal values of the parameter $p=(r_x,r_y,r_z,k_m )^*$ by minimizing the objective function: 

$L(p)=\alpha ∑_i|A^p (\nu_i )-A(\nu_i )|^2 +\beta ∑_i|\mathcal{A}_{true}^p (\nu_i )-\mathcal{A}_{true}^{ED} (\nu_i )|^2$ .  

Therefore, the optimization relied on the surrogates models for the FDTD simulation sketched in Fig. 4(a), as described earlier in the Methods section of the paper. This model is trained to predict the absorbance and the true absorption spectra $( A^p(\nu), \mathcal{A}^p_{true} (\nu) ) $ from the design parameters, $p$ . In addition, the encoder-decoder model, which takes $A(\nu)$ as input and it provides $A_{true}^ED(\nu)$, was used to infer a physically plausible true absorption spectrum conditioned on the target absorbance (see Fig. 3(a)).

$^* p$ corresponds to the three ellipsoid semi-axes and the $k_m$ parameter for scaling the imaginary refractive index of Matrigel, $k(\nu)$.

$^{**}$ Important note: For variables related to absorbance, we use the symbol $E$ to avoid confusion with $A$, which in this notebook denotes absorption. 

# Main packages

In [ ]:
# Common packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load matlab files
import scipy.io

# Neural Networks packages
import tensorflow as tf
from tensorflow.keras import models

# Auxiliary Plotting Functions

In [ ]:
def plot_figure(x,y,xlabel,ylabel,labels,**kwargs):
    import matplotlib.pyplot as plt
 
    xmin=kwargs.get('xmin')
    xmax=kwargs.get('xmax')
    ymin=kwargs.get('ymin')
    ymax=kwargs.get('ymax')
    title=kwargs.get('title')
    symbol_size=kwargs.get('symbol_size',0)
    line_width=kwargs.get('line_width',2)

    colors,line = [],[]
    colors = ["blue","green","red"]
    line = ["-","--","dotted"]
    dpi=110; figy=2;figx=1.5*figy
    fig, ax = plt.subplots(num=None, dpi=dpi,figsize=(figx,figy), facecolor='w', edgecolor='k')
    for i in range(0,len(y)):
        #print(i,x[i].shape,y[i].shape)
        ax.plot(np.squeeze(x[i]),np.squeeze(y[i]), color=colors[i],ls=line[i], label=labels[i],
                marker='.',markersize=symbol_size,linewidth=line_width)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_xlim(xmin,xmax)
    ax.set_ylim(ymin,ymax)
    #ax.set_ylabel("Scattering coefficients")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.2), mode="expand",
              ncol=2, fontsize=6, frameon=False)
    ax.set_title(title, fontsize=8, pad=20)
    #fig.tight_layout(rect=[0, 0, 1, 0.92])
    ax.grid()        
pass

# Load HeLa Cells Data

In [ ]:
path="./data/hela_cells/"

# Choose Amide 1 band. 
# Note that: 1552 cm-1 = Amide2 1648 cm-1 = Amide1
nuc_or_cell='nuc' # Choose 'nuc' or 'cell'
hela_name='hela3' # Choose 'hela1', 'hela2' or 'hela3'
k_max=['0p25',0.25]

cell_info={'name':hela_name,'k':k_max[1]}
if (nuc_or_cell=='nuc'):
	if(hela_name=='hela1'):
	 	cell_info['xyz']=[4.810,4.320,5.373]  # Grid estimation
	elif(hela_name=='hela2'):
	  	cell_info['xyz']= [10.565,6.890,2.304] #[10.493,6.715,2.28]->OLD
	else:
	  cell_info['xyz']=[7.602,15.357,1.448] # Grid estimation

if (nuc_or_cell=='cell'):
	if(hela_name=='hela1'):
	 cell_info['xyz']=[6.996,7.164,7.360]
	elif(hela_name=='hela2'):
	  cell_info['xyz']= [14.124,9.497,2.802] #[10.493,6.715,2.28]->OLD
	else:
	  cell_info['xyz']=[11.845,20.198,1.658] # Grid estimation

'''
Laod data here. Note that the data is in matlab format,
so we use scipy.io.loadmat to load it.
The data contains the frequency range (vec_wn),
the extinction (vec_a),
the normalized extinction (norm_exc) and
the absorption (frac_abs) for the chosen cell and k value.
'''
file_name=path+'model_'+hela_name+'_'+nuc_or_cell+'_k'+k_max[0]+'.fsp_spectra'
print(file_name)
hela_data = scipy.io.loadmat(file_name)
print(hela_data.keys())
vec_wn_hela=hela_data['vec_wn'] # Frequency range
nwav=len(vec_wn_hela)
wlength=np.squeeze(vec_wn_hela) # Frequency range
vec_a=hela_data['vec_a']   # Exctinction: raw data
norm_exc=hela_data['norm_a'] # Exctinction: Normalized data to Amine 1
frac_abs=hela_data['frac_abs']   # absorption: raw data

## Plotting Spectra

In [ ]:
import os
cell_names = ['hela1', 'hela2', 'hela3']
nuc_or_cell = 'cell' # Choose 'nuc' or 'cell'
k_tag = '0p25'
path = globals().get('path', './hela_cells/')
print(path)
spectra_all = {}
for name in cell_names:
    filename = f"model_{name}_{nuc_or_cell}_k{k_tag}.fsp_spectra"
    filepath = os.path.join(path, filename)

    data = scipy.io.loadmat(filepath)
    wlength_cell = np.squeeze(data['vec_wn'])
    spectra_all[name] = {
        'wlength': wlength_cell,
        'norm_exc': np.squeeze(data['norm_a']),
        'frac_abs': np.squeeze(data['frac_abs'])
    }

if len(spectra_all) == 0:
    raise FileNotFoundError(f"No spectra files found in {path} for {cell_names}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=False)

for name, spec in spectra_all.items():
    axes[0].plot(spec['wlength'], spec['norm_exc'], label=name)
axes[0].set_xlabel('wavenumber $(cm^{-1})$')
axes[0].set_ylabel('Normalized extinction')
axes[0].set_title('Extinction for all Hela cells')
axes[0].legend()
axes[0].grid(True)

for name, spec in spectra_all.items():
    axes[1].plot(spec['wlength'], spec['frac_abs'], label=name)
axes[1].set_xlabel('wavenumber $(cm^{-1})$')
axes[1].set_ylabel('Absorption')
axes[1].set_title('Absorption for all Hela cells')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Load Spectra and Parameters Used in the training of Encoder-Decoder and Surrogate Neural Network Models

In [ ]:
# Load the Training and Validation Datasets
path="./data/training_data/"
collect = scipy.io.loadmat(path+'fdtd_simulations.mat')

# Wavelengths
wlength=collect['vec_wn']

# Spectra
absorbance=collect['collect_spec_norm'].T
absorption=collect['collect_frac_abs'].T

# These will be used to remove 
# normalization from the Surrogate Neural Network outputs
max_absorbance = np.max(absorbance)
max_absorption = np.max(absorption)

print("Values of maximum absorption and absorbance from surrogate models:",max_absorption,max_absorbance)

In [ ]:
# Parameters (radii,material (k))
collect_params=collect['collect_params']

params=collect_params[0,0]
params_list=[p for p in params]
df_radii=pd.DataFrame(params_list[0])
df_material=pd.DataFrame(params_list[1])
materialno=params_list[2]

# INPUT NEURAL NETWORK
df_x=pd.concat([df_radii, df_material], axis=1, ignore_index=True)
df_x.columns=['rx','ry','rz','k']

max_param = np.array(df_x.max())

# These will be used to remove 
# normalization from the Surrogate Neural Network inputs
print("Maximum values for parameters:",max_param)

# Load Encoder-Decoder and Surrogate Neural Network Models

In [ ]:
import os

# True absorption from a "experimental" absorbance = Cell's absorbance
current_dir = os.getcwd()
print(f"Current working directory: {current_dir}")

model_name_1=os.path.join(current_dir, "encoder_decoder_network_model.keras")
print(f"Path for model_name_1: {model_name_1}")

# Absorbance and Absorption from p=paramters (regression model)
model_name_2=os.path.join(current_dir, "surrogate_neural_network_model_absorption.keras")
print(f"Path for model_name_2: {model_name_2}")
model_name_3=os.path.join(current_dir, "surrogate_neural_network_model_absorbance.keras")
print(f"Path for model_name_3: {model_name_3}")

try:
    model_AfromE=models.load_model(model_name_1)
    model_Ap=models.load_model(model_name_2) 
    model_Ep=models.load_model(model_name_3)

    model_AfromE.summary()
    model_Ap.summary()
    model_Ep.summary()
except ValueError as e:
    print(f"Error loading Keras models: {e}")
    print("This often indicates that the .keras model file is corrupted, not a valid Keras model format, or incompatible with the current Keras version.")
    print("Please verify the integrity and format of your model files at the specified paths.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    print("Please check your model files and environment setup.")

# Inverse Design Optimization

## Target: HeLa Cell

In [ ]:
nwav=wlength.shape[0]
x_target=np.zeros(shape=(1,4))
y_target=np.array(norm_exc).reshape((1,nwav))
y_target=tf.convert_to_tensor(y_target,dtype=tf.float32)
target_spectrum = y_target if isinstance(y_target, tf.Tensor) else tf.convert_to_tensor(np.array(y_target, dtype=np.float32), dtype=tf.float32)

## Descent Gradient Algorithm

### Definition of the Loss and the Optimization Function

In [ ]:
@tf.function
def combined_loss(param,
                  alfa_=1.0,
                  beta_=1.0,
                  norm_factor_absorbance=1.0,
                  norm_factor_absorption=1.0):  
    E_pred = norm_factor_absorbance * model_Ep(param, training=False)
    A_pred = norm_factor_absorption * model_Ap(param, training=False)
    A_from_E = model_AfromE(target_spectrum, training=False)
    loss_e = loss_fn(target_spectrum, E_pred)
    loss_a = loss_fn(A_pred, A_from_E)
    return alfa_ * loss_e + beta_ * loss_a, E_pred, A_pred, A_from_E

def optimize_inverse_design(initial_param,
                            learning_rate,
                            norm_factor_absorbance=1.0,
                            norm_factor_absorption=1.0):
    
    norm_factor_absorbance = tf.constant(norm_factor_absorbance,dtype=tf.float32)
    norm_factor_absorption = tf.constant(norm_factor_absorption,dtype=tf.float32)

    param_var = tf.Variable(initial_param, dtype=tf.float32)
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    best_loss = np.inf
    best_param = initial_param.copy()
    loss_history = []
    for step in range(steps):
        with tf.GradientTape() as tape:
            loss_value_tensor, _, _, _ = combined_loss(param_var, 
                                                       alfa_=alfa, 
                                                       beta_=beta,
                                                       norm_factor_absorbance=norm_factor_absorbance,
                                                       norm_factor_absorption=norm_factor_absorption)
        grads = tape.gradient(loss_value_tensor, param_var)
        if k_const:
            grads = tf.tensor_scatter_nd_update(grads, [[0, 3]], [0.0])
        optimizer.apply_gradients([(grads, param_var)])
        param_var.assign(tf.clip_by_value(param_var, 0.0, 1.0))
        loss_scalar = float(loss_value_tensor.numpy())
        loss_history.append(loss_scalar)
        if loss_scalar < best_loss:
            best_loss = loss_scalar
            best_param = param_var.numpy().copy()
        if step % 500 == 0 or step == steps - 1:
            print(f"step {step:5d} loss {loss_scalar:.6e} x_norm {param_var.numpy()[0]}")
    return best_param, best_loss, np.array(loss_history)

### Initial Parameters for Optimization

In [ ]:
# Alternative inverse design optimization with combined loss and parameter clipping
loss_fn = tf.keras.losses.MeanSquaredError()
alfa = 1.0
beta = 1.0
learning_rate = 1e-3
steps = 10000
k_const = False # Leaves the k-value constant

# Initial seed for optimization (normalized)
df_seed=np.array([5.0*1e-6,5.0*1e-6,5.0*1e-6,0.5])/max_param
x_start = np.clip(np.abs(df_seed.reshape(1, 4).astype(np.float32)), 0.0, 1.0)

### Optimization

In [ ]:
# Number of cases
M=10

# List for cost and parameters predicted (x)
cost_M,x_M=[],[]

# Loop over M
for m in range(0,M):
    print("Run m=",m)
    # Initial conditions    
    x_start=df_seed.reshape(1,4) +np.random.randn(1, 4)*1e-1 
    x_best, best_loss, loss_history = optimize_inverse_design(x_start,
                                                              learning_rate,
                                                              norm_factor_absorbance=max_absorbance,
                                                              norm_factor_absorption=max_absorption)
    print("best normalized x:", x_best[0])
    print("best physical x:", x_best[0] * max_param)
    print("best loss:", best_loss)

    plt.plot(loss_history)
    plt.yscale('log')
    plt.xlabel('iteration')
    plt.ylabel('loss')
    plt.title(f'Combined loss history alfa={alfa} beta={beta}')
    plt.show()

    cost_M.append(best_loss)
    x_M.append(x_best)

### Sorting the Results by Cost and Defining the Physical Parameters Predicted

In [ ]:
# Sort results by cost
sorted_indices = np.argsort(cost_M)
cost_M_sorted = [cost_M[i] for i in sorted_indices]
x_M_sorted = [x_M[i] for i in sorted_indices]

print("Best cost:", cost_M_sorted[0])
print("Best parameters (normalized):", x_M_sorted[0])
print("Best parameters (physical):", x_M_sorted[0] * max_param)
print("\nCell info=",cell_info)
x_M_sorted_physical = np.array([x * max_param for x in x_M_sorted]).squeeze()

# x_M_sorted_physical in microns by multiplying by 1e6 the three first parameters (radii) and keeping k as is
x_M_sorted_physical[:,:3] = x_M_sorted_physical[:,:3] * 1e6
print("All costs:", cost_M_sorted)
print("Shape of physical parameters:", x_M_sorted_physical.shape)
print("Physical parameters for all runs:\n", x_M_sorted_physical)

### Creating arrays for Absorbance and Absorption Best Predictions

In [ ]:
E_pred_best_M, A_pred_best_M = [], []

# Max values for normalization
x_M_sorted= np.array([x /max_param for x in x_M_sorted_physical]).squeeze()
x_M_sorted[:,0:3] = x_M_sorted[:,:3] * 1e-6 # Convert back to normalized parameters for the model input

for i in range(0,M):
    x_ = x_M_sorted[i].reshape(1, 4).astype(np.float32)
    print(f"Run {i}: Cost = {cost_M_sorted[i]:.6e}, Parameters (physical) = {x_M_sorted_physical[i]}")
    E_pred_best = max_absorbance * model_Ep(tf.convert_to_tensor(x_, dtype=tf.float32), training=False).numpy()
    A_pred_best = max_absorption * model_Ap(tf.convert_to_tensor(x_, dtype=tf.float32), training=False).numpy()
    E_pred_best_M.append(E_pred_best)
    A_pred_best_M.append(A_pred_best)

E_pred_best_M = np.array(E_pred_best_M).squeeze()
A_pred_best_M = np.array(A_pred_best_M).squeeze()

## Rendering Results

### Compare Target Spectra Against Optimizations

In [ ]:
### Compare target (experimental/FDTD) spectra against predictions
lon_onda_ini=wlength.min()
lon_onda_fin=wlength.max()
xplot=[wlength,wlength]
nwav=wlength.shape[0]
y_target=np.array(norm_exc).reshape((1,nwav))
y_target=tf.convert_to_tensor(y_target,dtype=tf.float32)
M=len(cost_M)    

# Experimental vs predicted spectrum for extinction and absorption
# Random choice between the M runs
i=np.random.randint(0,M)
print(f"Randomly selected run {i} with cost {cost_M[i]:.6e} and parameters (physical) {x_M_sorted_physical[i]}")
pred_param=np.round(np.array(x_M_sorted_physical[i]),3) # Run in previous cell

NN_pred_opt_E= E_pred_best_M[i] #model_Ep(tf.convert_to_tensor(x_M[i].reshape(1,4),dtype=tf.float32),training=False).numpy()
NN_pred_opt_A=A_pred_best_M[i] #model_Ap(tf.convert_to_tensor(x_M[i].reshape(1,4),dtype=tf.float32),training=False).numpy()
yplot_E=[np.squeeze(norm_exc),np.squeeze(NN_pred_opt_E)]    

yplot_A=[np.squeeze(frac_abs),np.squeeze(model_AfromE(y_target,training=False)),np.squeeze(NN_pred_opt_A)]       

true_param=cell_info['xyz']+[cell_info['k']]
added_title = (
        f"\nRef param.: {np.round(np.array(true_param)[:3], 3).tolist()} "
        f"{np.round(np.array(true_param)[3], 3):.3f}\n"
        f"NN pred. param.: {pred_param[:3]} "
        f"{pred_param[3]:.3f}"
    )


plot_figure(xplot,yplot_E,xlabel='wavevector$(cm^{-1})$',ylabel='Absorbance',labels=['HeLa (FDTD)','Pred. inv.desig.'],
            xmin=lon_onda_ini,xmax=lon_onda_fin,title=added_title) 

xplot=[wlength,wlength,wlength]
plot_figure(xplot,yplot_A,xlabel='wavevector$(cm^{-1})$',ylabel='Absorption',labels=['HeLa (FDTD)','From Exc. target (via NN)','Pred. inv.desig.'],
            xmin=lon_onda_ini,xmax=lon_onda_fin,title=added_title) 


### Results Summary and Prediction Selection Based on a Tolerance Around the Best Cost

In [ ]:
# Summary of the results
print("PREDICTIONS:")
print("Cell grid=",cell_info)
print("NN optimization - Max loss=",np.max(cost_M))
print("NN optimization - Min loss=",np.min(cost_M))
print("NN optimization best:")
print("Best pred. X,Y,Z=",x_M_sorted_physical[0][0:3])
print('Best pred. k=',x_M_sorted_physical[0][3])


# Select predictions within a certain tolerance of the best cost
tolerance = 1e-4
print(f"\nPredictions with cost within {tolerance} of the best cost:")
cost_ref = np.asarray(globals().get('cost_Me',
                                   np.asarray(cost_M_sorted if 'cost_M_sorted' in globals() else cost_M)))
best_cost = np.min(cost_ref)
mask = cost_ref <= best_cost + tolerance

x = x_M_sorted_physical[mask, 0]
y = x_M_sorted_physical[mask, 1]
z = x_M_sorted_physical[mask, 2]
k = x_M_sorted_physical[mask, 3]
cost_filtered = cost_ref[mask]

for idx, (xi, yi, zi, ki) in enumerate(zip(x, y, z, k)):
    print('\n', idx)
    print("Cost:", cost_filtered[idx])
    print("Pred. X,Y,Z=", xi, yi, zi)
    print('Pred. k=', ki)

### Maps of $r_x$ vs. $r_y$, etc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Info from cell frame (kc is not adjusted)
xc=cell_info['xyz'][0]
yc=cell_info['xyz'][1]
zc=cell_info['xyz'][2]
kc=cell_info['k']
# Create a 3D scatter plot
fig = plt.figure(figsize=(10, 8))

# Create XY projection
ax_xy = fig.add_axes([1.0, 0.6, 0.35, 0.35])
sc_xy = ax_xy.scatter(x, y, c=k, cmap='viridis',marker='x', s=200, alpha=0.8,label='p')
ax_xy.scatter(xc, yc, c=kc, cmap='viridis', s=100, alpha=1.0,edgecolors='red',marker='.',label='Ref.')
ax_xy.set_xlabel(r'$r_x(\mu m)$')
ax_xy.set_ylabel(r'$r_y(\mu m)$')
ax_xy.set_title('XY Projection')

# Create XZ projection
ax_xz = fig.add_axes([1.5, 0.1, 0.35, 0.35])
sc_xz = ax_xz.scatter(x, z, c=k, cmap='viridis',marker='x', s=200, alpha=0.8)
ax_xz.scatter(xc, zc, c=kc, cmap='viridis', s=100, alpha=1.0,edgecolors='red',marker='.')
ax_xz.set_xlabel(r'$r_x(\mu m)$')
ax_xz.set_ylabel(r'$r_z(\mu m)$')
ax_xz.set_title('XZ Projection')

# Create YZ projection
ax_yz = fig.add_axes([1.0, 0.1, 0.35, 0.35])
sc_yz = ax_yz.scatter(y, z, c=k, cmap='viridis',marker='x', s=200, alpha=0.8)
ax_yz.scatter(yc, zc, c=kc, cmap='viridis', s=100, alpha=1.0,edgecolors='red',marker='.')
ax_yz.set_xlabel(r'$r_y(\mu m)$')
ax_yz.set_ylabel(r'$r_z(\mu m)$')
ax_yz.set_title('YZ Projection')

cbar = plt.colorbar(sc_xy)
cbar.set_label('k parameter')
cbar.ax.invert_yaxis()  # Invert y-axis of color bar if you want higher k values at the top

#plt.tight_layout()
ax_xy.legend()
plt.show()